# LangChain + Geopack MCP — geocode then search

Teaches the **geocode → list** workflow:

1. `geopack_sdk_geocode_place` → WGS84 bbox
2. `geopack_sdk_list_datasets(bbox=..., data_type=...)`

Same discovery pattern as [`examples/langchain_geopack_geocode_workflow.py`](../examples/langchain_geopack_geocode_workflow.py), shown step-by-step in the notebook.

See also: [`examples/langchain_geopack_agent.py`](../examples/langchain_geopack_agent.py) for the minimal terminal sample.

In [ ]:
%pip install -q python-dotenv nest_asyncio
%pip install -q -e "../.[langchain]"

In [9]:
import sys
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
SDK_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "lib").is_dir() else NOTEBOOK_DIR
sys.path.insert(0, str(SDK_ROOT / "src"))
sys.path.insert(0, str(SDK_ROOT / "notebooks" / "lib"))

load_dotenv(SDK_ROOT / "notebooks" / ".env")
load_dotenv(SDK_ROOT / ".env")

from notebook_langchain import setup_notebook_paths, load_env

SDK_ROOT = setup_notebook_paths()
load_env(SDK_ROOT)

## Step 1 — Environment & MCP tools

In [10]:
import os
from notebook_langchain import notebook_mcp_langchain_tools, create_chat_model, GEOCODE_SYSTEM_PROMPT

assert os.getenv("GEOPACK_API_URL") and os.getenv("OPENAI_API_KEY")

if "mcp_ctx" in globals() and mcp_ctx is not None:
    await mcp_ctx.__aexit__(None, None, None)
mcp_ctx = notebook_mcp_langchain_tools()
tools, mcp_session, transport = await mcp_ctx.__aenter__()
print("Transport:", transport, "| tools:", len(tools))

Geopack SDK MCP: logged in as admin


Transport: inprocess | tools: 11


## Step 2 — Agent with geocode-first system prompt

The LLM is instructed to call **geocode** before **list** when the user mentions a place.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

llm = create_chat_model()
agent = create_agent(llm, tools, system_prompt=GEOCODE_SYSTEM_PROMPT, middleware=[SummarizationMiddleware(model=llm, max_history_length=10)])
print("System prompt requires geocode → list when a place is mentioned.")

System prompt requires geocode → list when a place is mentioned.


## Step 3 — Run prompt (Tehran rasters example)

In [12]:
USER_PROMPT = "Find raster datasets in the Tehran area since 2024."

print("User:", USER_PROMPT)
agent_result = await agent.ainvoke({"messages": USER_PROMPT})
messages = agent_result.get("messages", [])

User: Find raster datasets in the Tehran area since 2024.


HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://models.github.ai/inference/chat/completions "HTTP/1.1 413 Payload Too Large"


APIStatusError: Error code: 413 - {'error': {'code': 'tokens_limit_reached', 'message': 'Request body too large for gpt-4.1 model. Max size: 8000 tokens.', 'details': 'Request body too large for gpt-4.1 model. Max size: 8000 tokens.'}}

## Step 4 — Inspect tool chain (geocode → list)

In [ ]:
import json
from langchain_core.messages import ToolMessage
from notebook_langchain import print_tool_trace, last_assistant_text

print_tool_trace(messages, max_tools=12)

for msg in messages:
    if isinstance(msg, ToolMessage) and "bbox" in (msg.content or ""):
        try:
            geo = json.loads(msg.content)
            if "bbox" in geo:
                print("\nGeocode bbox:", geo.get("bbox"))
                print("Place:", geo.get("display_name"))
        except json.JSONDecodeError:
            pass

print("\n--- Assistant ---\n")
print(last_assistant_text(agent_result))

## Step 5 — HTML table + thumbnails

In [ ]:
from notebook_langchain import extract_list_datasets_from_messages
from display_datasets import display_datasets_rich

datasets = extract_list_datasets_from_messages(messages)
print(f"Datasets from list_datasets tool: {len(datasets)}")
await display_datasets_rich(datasets, mcp_session, max_rows=8)

In [ ]:
await mcp_ctx.__aexit__(None, None, None)